# XGBoost + MLflow — Suivi d'Expérience
### Potabilité de l'Eau

**Étapes couvertes :**
1. Configuration du serveur MLflow (port 5000)
2. Initialisation de l'expérience `experiment_water_quality`
3. Enregistrement complet du modèle XGBoost dans MLflow
4. Consultation des résultats

---
## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json
import subprocess
import time
import os
import urllib.request

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)
import xgboost as xgb

import mlflow
import mlflow.xgboost
from mlflow import MlflowClient

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED         = 42
MLFLOW_PORT  = 5000
MLFLOW_DB    = 'sqlite:///mlflow_water.db'
ARTIFACT_DIR = './mlflow_artifacts'
EXP_NAME     = 'experiment_water_quality'
MODEL_NAME   = 'WaterQualityXGBoost'
C_NEG, C_POS = '#F44336', '#2196F3'

np.random.seed(SEED)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print(f'XGBoost : {xgb.__version__}')
print(f'MLflow  : {mlflow.__version__}')


---
## 2. Configuration du Serveur MLflow (port 5000)

In [ ]:
server_cmd = [
    'mlflow', 'server',
    '--backend-store-uri', MLFLOW_DB,
    '--default-artifact-root', ARTIFACT_DIR,
    '--host', '0.0.0.0',
    '--port', str(MLFLOW_PORT),
]

print('Demarrage du serveur MLflow...')
print(f'Commande : {" ".join(server_cmd)}')

mlflow_server = subprocess.Popen(
    server_cmd,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(2)

if mlflow_server.poll() is None:
    print(f'Serveur MLflow actif  (PID : {mlflow_server.pid})')
    print(f'   Interface web  -> http://localhost:{MLFLOW_PORT}')
    print(f'   Backend store  -> {MLFLOW_DB}')
    print(f'   Artifacts root -> {ARTIFACT_DIR}')
    print(f'   Verifier port  -> netstat -ano | findstr :{MLFLOW_PORT}')
else:
    print('Le serveur a deja ete lance ou le port est occupe.')
    print(f'   -> Verifier : netstat -ano | findstr :{MLFLOW_PORT}')


In [ ]:
# Connexion au serveur MLflow avec retry
# Endpoint /api/2.0/mlflow/experiments/search disponible des le demarrage

HEALTH_ENDPOINT = f"http://localhost:{MLFLOW_PORT}/api/2.0/mlflow/experiments/search"
TRACKING_URI    = None
MAX_RETRIES     = 15

for attempt in range(1, MAX_RETRIES + 1):
    try:
        urllib.request.urlopen(HEALTH_ENDPOINT, timeout=1)
        TRACKING_URI = f"http://localhost:{MLFLOW_PORT}"
        print(f"Connecte au serveur MLflow (tentative {attempt}/{MAX_RETRIES})")
        print(f"   Tracking URI : {TRACKING_URI}")
        break
    except Exception:
        print(f"   Tentative {attempt}/{MAX_RETRIES} — serveur pas encore pret...")
        time.sleep(1)

if TRACKING_URI is None:
    TRACKING_URI = MLFLOW_DB
    print(f"Serveur HTTP non disponible apres {MAX_RETRIES}s — fallback SQLite")
    print(f"   Tracking URI : {TRACKING_URI}")

mlflow.set_tracking_uri(TRACKING_URI)
print(f"\n   URI active : {mlflow.get_tracking_uri()}")


---
## 3. Initialisation de l'Expérience MLflow

In [ ]:
client = MlflowClient(tracking_uri=TRACKING_URI)

experiment = client.get_experiment_by_name(EXP_NAME)

if experiment is None:
    experiment_id = client.create_experiment(
        name              = EXP_NAME,
        artifact_location = ARTIFACT_DIR,
        tags = {
            'project'    : 'Water Quality Prediction',
            'dataset'    : 'water_potability.csv',
            'task'       : 'Binary Classification',
            'framework'  : 'XGBoost',
            'created_by' : 'notebook',
        }
    )
    print(f'Experience creee : "{EXP_NAME}"')
else:
    experiment_id = experiment.experiment_id
    print(f'Experience recuperee : "{EXP_NAME}"')

experiment = client.get_experiment(experiment_id)
mlflow.set_experiment(EXP_NAME)

print()
print(f'   ID                : {experiment.experiment_id}')
print(f'   Nom               : {experiment.name}')
print(f'   Artifact location : {experiment.artifact_location}')
print(f'   Statut            : {experiment.lifecycle_stage}')
print(f'   Tags              : {experiment.tags}')


In [ ]:
# Description markdown de l'experience — visible dans l'onglet Experiments
client.set_experiment_tag(
    experiment_id,
    "mlflow.note.content",
    """
## Water Quality Prediction

**Objectif** : Classification binaire — predire la potabilite de l'eau.

**Dataset** : 3276 echantillons, 9 features physico-chimiques.

**Pipeline** :
- Imputation mediane par classe (ph, Sulfate, Trihalomethanes)
- Winsorisation 1%–99%
- SMOTE pour reequilibrage (61/39 → 50/50)
- RobustScaler

**Modele** : XGBoost 500 arbres — ROC-AUC val = 0.8765
    """
)
print("Description ajoutee a l'experience")

---
## 4. Chargement des Artefacts d'Entraînement

In [ ]:
ARTIFACTS_DIR = "model_artifacts"

model = xgb.XGBClassifier()
model.load_model(f"{ARTIFACTS_DIR}/xgboost_model.json")

scaler       = joblib.load(f"{ARTIFACTS_DIR}/robust_scaler.pkl")
X_val_sc     = np.load(f"{ARTIFACTS_DIR}/X_val_sc.npy")
y_val        = pd.Series(np.load(f"{ARTIFACTS_DIR}/y_val.npy"))
y_pred       = np.load(f"{ARTIFACTS_DIR}/y_pred.npy")
y_pred_prob  = np.load(f"{ARTIFACTS_DIR}/y_pred_prob.npy")
cv_scores    = np.load(f"{ARTIFACTS_DIR}/cv_scores.npy")

with open(f"{ARTIFACTS_DIR}/metadata.json") as f:
    metadata = json.load(f)

features        = metadata["features"]
xgb_params      = metadata["params"]
val_metrics_ref = metadata["metrics"]

# Courbe apprentissage — train vs val logloss par iteration
with open(f"{ARTIFACTS_DIR}/evals_result.json") as f:
    evals_result = json.load(f)

# n_estimators recupere depuis les metadonnees (attribut non expose apres load_model)
n_estimators_display = xgb_params.get('n_estimators', model.get_params().get('n_estimators', '?'))

print(f"Modele et artefacts charges depuis {ARTIFACTS_DIR}/")
print(f"   Modele     : XGBoostClassifier ({n_estimators_display} arbres)")
print(f"   Features   : {features}")
print(f"   Val size   : {len(X_val_sc)} echantillons")
print(f"   CV ROC-AUC : {metadata['cv_mean']:.4f} +/- {metadata['cv_std']:.4f}")


---
## 5. Démarrage du Run MLflow & Enregistrement du Modèle

In [ ]:
print("   Hyperparametres charges depuis metadata.json :")
for k, v in xgb_params.items():
    print(f"   {k:25s}: {v}")


In [ ]:
# Verification de run existant (evite les doublons)
RUN_NAME        = "XGBoost_SMOTE_RobustScaler_v1"
existing_run_id = None

existing_runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string=f"tags.mlflow.runName = '{RUN_NAME}' and attributes.status = 'FINISHED'",
    max_results=1,
)

if existing_runs:
    existing_run_id = existing_runs[0].info.run_id
    print(f"Run '{RUN_NAME}' deja existant (ID: {existing_run_id[:20]}...)")
    print("   Creation d'un nouveau run numerote pour eviter les doublons.")
    all_similar = client.search_runs(
        experiment_ids=[experiment_id],
        filter_string=f"tags.mlflow.runName LIKE '{RUN_NAME}%'",
    )
    RUN_NAME = f"{RUN_NAME}_run{len(all_similar) + 1}"
    print(f"   Nouveau nom de run : {RUN_NAME}")
else:
    print(f"Aucun run existant pour '{RUN_NAME}' — creation en cours...")


In [ ]:
TEMP_FILES = ["feature_importance.csv", "classification_report.txt", "dashboard_xgboost.png"]

with mlflow.start_run(
    experiment_id = experiment_id,
    run_name      = RUN_NAME
) as run:

    run_id = run.info.run_id
    print(f"Run demarre")
    print(f"   Run ID       : {run_id}")
    print(f"   Experience   : {EXP_NAME} (id={experiment_id})")
    print()

    # 1. TAGS
    mlflow.set_tags({
        "model_type"      : "XGBoostClassifier",
        "dataset"         : "water_potability.csv",
        "dataset_version" : "v1_winsorized_imputed",
        "n_classes"       : "2",
        "class_labels"    : "0=NonPotable,1=Potable",
        "preprocessing"   : "imputation_mediane + winsorisation_1pct99pct",
        "resampling"      : "SMOTE",
        "scaler"          : "RobustScaler",
        "split"           : "80_20_stratifie",
        "task"            : "binary_classification",
        "target"          : "Potability",
        "source_notebook" : "water_xgboost.ipynb",
        "python_env"      : "requirements.txt",
    })
    print("   Tags enregistres")

    # 2. PARAMETRES
    mlflow.log_params(xgb_params)
    mlflow.log_params({
        "n_train_raw"      : metadata["n_train_raw"],
        "n_train_smote"    : metadata["n_train_smote"],
        "n_val"            : metadata["n_val"],
        "n_features"       : len(features),
        "winsorize_limits" : "1pct_99pct",
        "imputation"       : "median_by_group",
    })
    print("   Parametres enregistres")

    # 3. METRIQUES
    val_metrics = {
        "val_accuracy"  : accuracy_score(y_val, y_pred),
        "val_f1"        : f1_score(y_val, y_pred),
        "val_precision" : precision_score(y_val, y_pred),
        "val_recall"    : recall_score(y_val, y_pred),
        "val_roc_auc"   : roc_auc_score(y_val, y_pred_prob),
        "val_pr_auc"    : average_precision_score(y_val, y_pred_prob),
    }
    mlflow.log_metrics(val_metrics)
    mlflow.log_metrics({
        "cv_roc_auc_mean" : cv_scores.mean(),
        "cv_roc_auc_std"  : cv_scores.std(),
        "cv_roc_auc_min"  : cv_scores.min(),
        "cv_roc_auc_max"  : cv_scores.max(),
    })
    print("   Metriques enregistrees")

    # Metriques par iteration — dessine les courbes dans l'UI
    train_loss = evals_result["validation_0"]["logloss"]
    val_loss   = evals_result["validation_1"]["logloss"]
    for step, (tl, vl) in enumerate(zip(train_loss, val_loss)):
        mlflow.log_metrics({
            "train_logloss": tl,
            "val_logloss"  : vl,
        }, step=step)
    print("   Courbes logloss enregistrees (" + str(len(train_loss)) + " iterations)")

    # 4. ARTEFACTS
    imp_df = pd.DataFrame({
        "feature"    : features,
        "importance" : model.feature_importances_
    }).sort_values("importance", ascending=False)
    imp_df.to_csv("feature_importance.csv", index=False)
    mlflow.log_artifact("feature_importance.csv", artifact_path="reports")

    report = classification_report(y_val, y_pred,
                target_names=["Non potable", "Potable"])
    with open("classification_report.txt", "w") as f:
        lines_out = ["RAPPORT DE CLASSIFICATION - VALIDATION SET", "="*50, report,
                     "METRIQUES DETAILLEES :"]
        for k, v in val_metrics.items():
            lines_out.append(f"  {k}: {v:.4f}")
        lines_out.append(f"  cv_roc_auc: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
        f.write(chr(10).join(lines_out))
    mlflow.log_artifact("classification_report.txt", artifact_path="reports")

    # CSV metriques par classe
    from sklearn.metrics import classification_report
    report_dict = classification_report(
        y_val, y_pred,
        target_names=["Non potable", "Potable"],
        output_dict=True
    )
    metrics_per_class = pd.DataFrame(report_dict).T
    metrics_per_class.to_csv("per_class_metrics.csv")
    mlflow.log_artifact("per_class_metrics.csv", artifact_path="reports")
    if os.path.exists("per_class_metrics.csv"):
        os.remove("per_class_metrics.csv")

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()

    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=["Non potable", "Potable"],
                yticklabels=["Non potable", "Potable"],
                linewidths=1.5, linecolor="white", cbar=False,
                annot_kws={"size": 13, "weight": "bold"})
    axes[0].set_title("Matrice de Confusion", fontweight="bold")
    axes[0].set_ylabel("Reel"); axes[0].set_xlabel("Predit")

    fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
    axes[1].plot(fpr, tpr, color=C_POS, lw=2.5, label=f"AUC={val_metrics['val_roc_auc']:.4f}")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    axes[1].fill_between(fpr, tpr, alpha=0.08, color=C_POS)
    axes[1].set_title("Courbe ROC", fontweight="bold")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].legend()

    prec_c, rec_c, _ = precision_recall_curve(y_val, y_pred_prob)
    axes[2].plot(rec_c, prec_c, color="#FF7043", lw=2.5, label=f"AP={val_metrics['val_pr_auc']:.4f}")
    axes[2].fill_between(rec_c, prec_c, alpha=0.08, color="#FF7043")
    axes[2].set_title("Precision-Rappel", fontweight="bold")
    axes[2].set_xlabel("Rappel"); axes[2].set_ylabel("Precision"); axes[2].legend()

    imp_sorted = imp_df.sort_values("importance")
    colors_fi  = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(features)))
    axes[3].barh(imp_sorted["feature"], imp_sorted["importance"],
                 color=colors_fi, edgecolor="white")
    axes[3].set_title("Importance des Variables", fontweight="bold")

    for lbl, color, name in [(0, C_NEG, "Non potable"), (1, C_POS, "Potable")]:
        mask = y_val == lbl
        axes[4].hist(y_pred_prob[mask], bins=30, alpha=0.6,
                     color=color, label=name, density=True)
    axes[4].axvline(0.5, color="black", linestyle="--", lw=1.5)
    axes[4].set_title("Distribution des Probabilites", fontweight="bold")
    axes[4].legend()

    folds = [f"Fold {i+1}" for i in range(len(cv_scores))]
    bars  = axes[5].bar(folds, cv_scores, color="#7E57C2", edgecolor="white", width=0.6)
    axes[5].bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
    axes[5].axhline(cv_scores.mean(), color="#E53935", linestyle="--", lw=2,
                    label=f"Moy={cv_scores.mean():.4f}")
    axes[5].set_ylim(0.8, 1.0)
    axes[5].set_title("Validation Croisee 5-Fold", fontweight="bold")
    axes[5].set_ylabel("ROC-AUC"); axes[5].legend()

    plt.suptitle("Dashboard XGBoost — Potabilite de l'Eau",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("dashboard_xgboost.png", bbox_inches="tight", dpi=150)
    plt.show()
    mlflow.log_artifact("dashboard_xgboost.png", artifact_path="figures")
    print("   Artefacts enregistres (CSV, TXT, PNG)")

    for tmp in TEMP_FILES:
        if os.path.exists(tmp):
            os.remove(tmp)
    print("   Fichiers temporaires nettoyes")

    # 5. ENREGISTREMENT DU MODELE
    mlflow.xgboost.log_model(
        model,
        artifact_path         = "xgboost_model",
        registered_model_name = MODEL_NAME,
        input_example         = pd.DataFrame(X_val_sc[:3], columns=features),
    )
    print(f"   Modele enregistre dans le Registry : {MODEL_NAME}")
    print()
    print(f"Run termine avec succes")
    print(f"   -> http://localhost:{MLFLOW_PORT}/#/experiments/{experiment_id}/runs/{run_id}")


---
## 6. Consultation des Résultats via MLflow

In [ ]:
run_data = client.get_run(run_id)

print('╔══════════════════════════════════════════════════════╗')
print('║           RESUME DU RUN — MLflow                    ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Run ID       : {run_id[:20]}...              ║')
print(f'║  Experience   : {EXP_NAME}   ║')
print(f'║  Statut       : {run_data.info.status}                              ║')
print('╠══════════════════════════════════════════════════════╣')
print('║      METRIQUES DE VALIDATION :                       ║')
for k, v in run_data.data.metrics.items():
    print(f'║    {k:28s} : {v:.4f}          ║')
print('╠══════════════════════════════════════════════════════╣')
print('║      HYPERPARAMETRES PRINCIPAUX :                    ║')
key_params = ['n_estimators', 'max_depth', 'learning_rate', 'subsample',
              'colsample_bytree', 'reg_alpha', 'reg_lambda']
for k in key_params:
    v = run_data.data.params.get(k, 'N/A')
    print(f'║    {k:28s} : {v:<20s}      ║')
print('╠══════════════════════════════════════════════════════╣')
print('║      TAGS :                                          ║')
for k, v in run_data.data.tags.items():
    if not k.startswith('mlflow.'):
        print(f'║    {k:28s} : {str(v):<20s}      ║')
print('╚══════════════════════════════════════════════════════╝')


In [ ]:
print(f'\n Model Registry — "{MODEL_NAME}" :')
try:
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    for v in versions:
        print(f'  Version {v.version}')
        print(f'    Status  : {v.status}')
        print(f'    Run ID  : {v.run_id[:20]}...')
        print(f'    Source  : {v.source}')
except Exception as e:
    print(f'  Info registry : {e}')

print(f'\n Interface MLflow disponible sur :')
print(f'   http://localhost:{MLFLOW_PORT}')
print(f'   -> Onglet "Experiments" : {EXP_NAME}')
print(f'   -> Onglet "Models"      : {MODEL_NAME}')


In [ ]:
# Assigner alias "champion" a la version en production
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if versions:
    latest_version = versions[0].version
    try:
        client.set_registered_model_alias(
            name    = MODEL_NAME,
            alias   = "champion",
            version = latest_version,
        )
        print(f"Alias 'champion' -> {MODEL_NAME} v{latest_version}")
        print(f"   Charger avec : mlflow.xgboost.load_model('models:/{MODEL_NAME}@champion')")
    except Exception as e:
        print(f"Alias non supporte par cette version MLflow : {e}")

In [ ]:
model_uri    = f'runs:/{run_id}/xgboost_model'
loaded_model = mlflow.xgboost.load_model(model_uri)

sample        = pd.DataFrame(X_val_sc[:5], columns=features)
predictions   = loaded_model.predict(X_val_sc[:5])
probabilities = loaded_model.predict_proba(X_val_sc[:5])[:, 1]

print('Inference depuis le modele charge via MLflow :')
result = sample.copy()
result['Potability_reelle']  = y_val.values[:5]
result['Potability_predite'] = predictions
result['Proba_potable']      = probabilities.round(4)
result[['Potability_reelle', 'Potability_predite', 'Proba_potable']]


In [ ]:
# Decommenter pour stopper le serveur en fin de session
# mlflow_server.terminate()
# print('Serveur MLflow arrete')
print('Serveur MLflow toujours actif sur http://localhost:5000')
print("   Pour l'arreter : mlflow_server.terminate()")
